In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , GridSearchCV , KFold , cross_val_score
from sklearn.linear_model import LinearRegression 
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score , mean_squared_error , mean_absolute_error
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.base import BaseEstimator , TransformerMixin
import optuna
from lightgbm import LGBMRegressor
import mlflow
import dagshub
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate , plot_slice

In [2]:
dagshub.init(repo_owner="mridul0010" , repo_name="food-delivery-time-prediction" , mlflow=True)

Accessing as mridul0010

Initialized MLflow to track repo "mridul0010/food-delivery-time-prediction"

Repository mridul0010/food-delivery-time-prediction initialized!

In [3]:
mlflow.set_tracking_uri("https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow")

In [4]:
mlflow.set_experiment("4. Model Selection")

<Experiment: artifact_location='mlflow-artifacts:/e4b0fb1e261d4dc7a0a2eeb8fb8f24e9', creation_time=1783251238084, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1783251238084, lifecycle_stage='active', name='4. Model Selection', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [5]:
df = pd.read_csv('../data/Cleaned Delivery Dataset.csv')

In [6]:
pd.set_option('display.max_columns' , None)

In [7]:
df.drop(columns=['Delivery_Agent'] , inplace=True)

In [8]:
df['Order_Datetime'] = pd.to_datetime(df['Order_Datetime'])
df['Pickup_Datetime'] = pd.to_datetime(df['Pickup_Datetime'])

In [9]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.rating_bins = None

    def fit(self, X, y=None):
        X = X.copy()
        
        # Store bins for consistent transformation
        self.rating_bins = pd.qcut(
            X['Delivery_person_Ratings'],
            q=3,
            retbins=True,
            duplicates='drop'
        )[1]
        
        return self

    def transform(self, X):
        X = X.copy()

        # Datetime conversion
        X['Order_Datetime'] = pd.to_datetime(X['Order_Datetime'], errors='coerce')
        X['Pickup_Datetime'] = pd.to_datetime(X['Pickup_Datetime'], errors='coerce')

        # Rating group
        X["delivery_rating_group"] = pd.cut(
            X['Delivery_person_Ratings'],
            bins=self.rating_bins,
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

        # Age group
        X["age_group"] = pd.cut(
            X['Delivery_person_Age'],
            bins=[14, 25, 35, 60],
            labels=['Young', 'Adult', 'Senior']
        )

        # Distance group
        X["distance_group"] = pd.cut(
            X['distance_km'],
            bins=[0, 5, 10, 25],
            labels=['Short Distance', 'Medium Distance', 'Long Distance']
        )

        # Time features
        X['Prep_Time(min)'] = (
            X['Pickup_Datetime'] - X['Order_Datetime']
        ).dt.total_seconds() / 60

        X['Order_hour'] = X['Order_Datetime'].dt.hour
        X['Order_day'] = X['Order_Datetime'].dt.day_name()

        X['isWeekend'] = X['Order_day'].isin(["Saturday", "Sunday"]).astype(int)

        X['Time_Of_Day'] = pd.cut(
            X['Order_hour'],
            bins=[0, 6, 12, 18, 24],
            labels=["Night", "Morning", "Afternoon", "Evening"],
            include_lowest=True
        )

        # Drop raw datetime
        X = X.drop(columns=['Order_Datetime', 'Pickup_Datetime'], errors='ignore')

        return X

    def get_feature_names_out(self , input_features = None):
        return input_features

In [10]:
X = df.drop(columns=['Time_taken (min)'])
y = df['Time_taken (min)']

In [11]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [12]:
X

,City,Zone,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,distance_km,Order_Datetime,Pickup_Datetime,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City_Type
0,Dehradun,Zone17,36,4.2,30.327968,78.046106,30.397968,78.116106,10.28,2022-12-02 21:55:00,2022-12-02 22:10:00,Fog,Jam,Good,Snack,motorcycle,3,No,Metropolitian
1,Kochi,Zone16,21,4.7,10.003064,76.307589,10.043064,76.347589,6.24,2022-02-13 14:55:00,2022-02-13 15:05:00,Stormy,High,Average,Meal,motorcycle,1,No,Metropolitian
2,Pune,Zone13,23,4.7,18.562450,73.916619,18.652450,74.006619,13.79,2022-04-03 17:30:00,2022-04-03 17:40:00,Sandstorms,Medium,Average,Drinks,scooter,1,No,Metropolitian
3,Ludhiana,Zone15,34,4.3,30.899584,75.809346,30.919584,75.829346,2.93,2022-02-13 09:20:00,2022-02-13 09:30:00,Sandstorms,Low,poor,Buffet,motorcycle,0,No,Metropolitian
4,Kanpur,Zone14,24,4.7,26.463504,80.372929,26.593504,80.502929,19.40,2022-02-14 19:50:00,2022-02-14 20:05:00,Fog,Jam,Average,Snack,scooter,1,No,Metropolitian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39637,Bangalore,Zone16,28,4.9,13.029198,77.570997,13.059198,77.600997,4.66,2022-03-30 21:55:00,2022-03-30 22:05:00,Sandstorms,Jam,Average,Meal,scooter,1,No,Metropolitian
39638,Ranchi,Zone16,35,4.2,23.371292,85.327872,23.481292,85.437872,16.60,2022-08-03 21:45:00,2022-08-03 21:55:00,Windy,Jam,Good,Drinks,motorcycle,1,No,Metropolitian
39639,Chennai,Zone08,30,4.9,13.022394,80.242439,13.052394,80.272439,4.66,2022-11-03 23:50:00,2022-11-04 00:05:00,Cloudy,Low,Average,Drinks,scooter,0,No,Metropolitian
39640,Coimbatore,Zone11,20,4.7,11.001753,76.986241,11.041753,77.026241,6.23,2022-07-03 13:35:00,2022-07-03 13:40:00,Cloudy,High,poor,Snack,motorcycle,1,No,Metropolitian


In [13]:
numerical = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 
       'multiple_deliveries', 'distance_km', 'Prep_Time(min)', 'Order_hour',
       'isWeekend']

ohe_col = ['City', 'Zone', 'Weather_conditions',
       'Type_of_order', 'Type_of_vehicle',
       'City_Type','Order_day', 'Time_Of_Day']

ordinal_col = ['Road_traffic_density' , 'Vehicle_condition' , 'Festival',
       'delivery_rating_group' , 'age_group' , 'distance_group']

In [14]:
Road_traffic_density = ['Low', 'Medium', 'High', 'Jam']
Vehicle_condition = ['poor', 'Average', 'Good' , 'Excellent']
Festival = ['No', 'Yes']
delivery_rating_group = ['Low', 'Medium', 'High']
age_group = ['Young', 'Adult', 'Senior']         
distance_group = ['Short Distance', 'Medium Distance', 'Long Distance']

In [15]:
transformer = ColumnTransformer(
    transformers=[
        ("ohe" , OneHotEncoder(sparse_output=False , handle_unknown='ignore') , ohe_col),
        ('oe' , OrdinalEncoder(categories=[
            Road_traffic_density , Vehicle_condition,
            Festival , delivery_rating_group,
            age_group , distance_group
        ]) , ordinal_col),
        ('Scaling' , StandardScaler() , numerical)
    ],remainder='passthrough'
)

In [16]:
pipeline = Pipeline(
    [
        ("FeatureEngineering" , FeatureEngineering()),
        ("ColumnTransformer" , transformer)
    ]
) 

In [17]:
X_train_trans = pipeline.fit_transform(X_train)
X_test_trans = pipeline.transform(X_test)

In [18]:
pt = PowerTransformer()

In [19]:
def objective(trial):
    with mlflow.start_run(nested=True):
        model_name = trial.suggest_categorical("model",["SVM","RF","KNN","GB","XGB","LGBM"])

        if model_name == "SVM":
            kernel_svm = trial.suggest_categorical("kernel_svm",["linear","poly","rbf"])
            if kernel_svm == "linear":
                c_linear = trial.suggest_float("c_linear",0,10)
                model = SVR(C=c_linear,kernel="linear")

            elif kernel_svm == "poly":
                c_poly = trial.suggest_float("c_poly",0,10)
                degree_poly = trial.suggest_int("degree_poly",1,5)
                model = SVR(C=c_poly,degree=degree_poly,
                            kernel="poly")

            else:
                c_rbf = trial.suggest_float("c_rbf",0,100)
                gamma_rbf = trial.suggest_float("gamma_rbf",0,10)
                model = SVR(C=c_rbf,gamma=gamma_rbf,
                            kernel="rbf")

        elif model_name == "RF":
            n_estimators_rf = trial.suggest_int("n_estimators_rf",10,200)
            max_depth_rf = trial.suggest_int("max_depth_rf",2,20)
            rf = RandomForestRegressor(n_estimators=n_estimators_rf,
                                        max_depth=max_depth_rf,
                                        random_state=42,
                                        n_jobs=-1)
            model = TransformedTargetRegressor(regressor=rf , transformer=pt)

        elif model_name == "GB":
            n_estimators_gb = trial.suggest_int("n_estimators_gb",10,200)
            learning_rate_gb = trial.suggest_float("learning_rate_gb",0,1)
            max_depth_gb = trial.suggest_int("max_depth_gb",2,20)
            gb = GradientBoostingRegressor(n_estimators=n_estimators_gb,
                                                learning_rate=learning_rate_gb,
                                                max_depth=max_depth_gb,
                                                random_state=42)
            model = TransformedTargetRegressor(regressor=gb , transformer=pt)

        elif model_name == "KNN":
            n_neighbors_knn = trial.suggest_int("n_neighbors_knn",1,25)
            weights_knn = trial.suggest_categorical("weights_knn",["uniform","distance"])
            knn = KNeighborsRegressor(n_neighbors=n_neighbors_knn,
                                        weights=weights_knn,n_jobs=-1)
            model = TransformedTargetRegressor(regressor=knn , transformer=pt)

        elif model_name == "XGB":
            n_estimators_xgb = trial.suggest_int("n_estimators_xgb",10,200)
            learning_rate_xgb = trial.suggest_float("learning_rate_xgb",0.1,0.5)
            max_depth_xgb = trial.suggest_int("max_depth_xgb",2,20)
            xgb = XGBRegressor(n_estimators=n_estimators_xgb,
                                    learning_rate=learning_rate_xgb,
                                    max_depth=max_depth_xgb,
                                    random_state=42,
                                    n_jobs=-1)
            model = TransformedTargetRegressor(regressor=xgb , transformer=pt)

        elif model_name == "LGBM":
            n_estimators_lgbm = trial.suggest_int("n_estimators_lgbm",10,200)
            learning_rate_lgbm = trial.suggest_float("learning_rate_lgbm",0.1,0.5)
            max_depth_lgbm = trial.suggest_int("max_depth_lgbm",2,20)
            lgbm = LGBMRegressor(n_estimators=n_estimators_lgbm,
                                    learning_rate=learning_rate_lgbm,
                                    max_depth=max_depth_lgbm,
                                    random_state=42)
            model = TransformedTargetRegressor(regressor=lgbm , transformer=pt)


        model.fit(X_train_trans , y_train)


        # log params
        mlflow.log_params(trial.params)
        mlflow.log_param("model_type" , model_name)

        

        # get the predictions
        y_pred_test = model.predict(X_test_trans)

        # calculate the error
        error = mean_absolute_error(y_test,y_pred_test)

        # log model_name
        mlflow.log_param("model",model_name)

        # log error
        mlflow.log_metric("MAE",error)

        return error

In [20]:
# create optuna study
study = optuna.create_study(direction="minimize",study_name="model_selection")

with mlflow.start_run(run_name="Best Model") as parent:
    # optimize the objective function
    study.optimize(objective,n_trials=30,n_jobs=-1)

    # log the best parameters
    mlflow.log_params(study.best_params)

    # log the best score
    mlflow.log_metric("best_score",study.best_value)

[I 2026-07-05 20:33:51,212] A new study created in memory with name: model_selection


🏃 View run clean-rook-914 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/4/runs/392d6868a6134b2eaa7a93c8f7292370
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/4


[I 2026-07-05 20:34:16,714] Trial 8 finished with value: 4.782582107120026 and parameters: {'model': 'RF', 'n_estimators_rf': 20, 'max_depth_rf': 4}. Best is trial 8 with value: 4.782582107120026.


🏃 View run youthful-bass-38 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/4/runs/3cf2b512ceeb45be8888d27806c22b5d
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/4
🏃 View run classy-lamb-534 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/4/runs/4ce6ff3a42ba47ed93ec2e091027d81f
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/4


C:\Users\LeoML\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


...

🏃 View run salty-shrimp-597 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/4/runs/b944da2ed99f4d34aabb7d3e96088b2a
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/4


[I 2026-07-05 20:52:47,650] Trial 20 finished with value: 3.766438265497936 and parameters: {'model': 'SVM', 'kernel_svm': 'poly', 'c_poly': 7.538524998480884, 'degree_poly': 3}. Best is trial 1 with value: 3.0475403643017533.


🏃 View run funny-snake-497 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/4/runs/e37e6c5ceddf4be989f8eb09c30d5590
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/4


[I 2026-07-05 21:09:00,581] Trial 23 finished with value: 4.101604707753101 and parameters: {'model': 'SVM', 'kernel_svm': 'poly', 'c_poly': 8.311534140296699, 'degree_poly': 5}. Best is trial 1 with value: 3.0475403643017533.


🏃 View run Best Model at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/4/runs/9339610540884375b28fd2e0e5a6eb94
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/4


In [21]:
study.best_value

3.0475403643017533

In [22]:
study.best_params

{'model': 'RF', 'n_estimators_rf': 134, 'max_depth_rf': 13}

In [23]:
study.trials_dataframe().head()

,number,value,datetime_start,datetime_complete,duration,params_c_linear,params_c_poly,params_c_rbf,params_degree_poly,params_gamma_rbf,params_kernel_svm,params_learning_rate_gb,params_learning_rate_lgbm,params_learning_rate_xgb,params_max_depth_gb,params_max_depth_lgbm,params_max_depth_rf,params_max_depth_xgb,params_model,params_n_estimators_gb,params_n_estimators_lgbm,params_n_estimators_rf,params_n_estimators_xgb,params_n_neighbors_knn,params_weights_knn,state
0,0,4.470294,2026-07-05 20:33:52.275037,2026-07-05 20:36:00.750280,0 days 00:02:08.475243,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KNN,NaN,NaN,NaN,NaN,12.0,distance,COMPLETE
1,1,3.047540,2026-07-05 20:33:52.275830,2026-07-05 20:35:08.088966,0 days 00:01:15.813136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.0,NaN,RF,NaN,NaN,134.0,NaN,NaN,NaN,COMPLETE
2,2,3.047806,2026-07-05 20:33:52.276544,2026-07-05 20:37:08.957801,0 days 00:03:16.681257,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.333788,NaN,NaN,11.0,NaN,NaN,LGBM,NaN,35.0,NaN,NaN,NaN,NaN,COMPLETE
3,3,4.567463,2026-07-05 20:33:52.276811,2026-07-05 20:44:22.579407,0 days 00:10:30.302596,5.874607,NaN,NaN,NaN,NaN,linear,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SVM,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,3.050770,2026-07-05 20:33:52.277376,2026-07-05 20:37:04.749278,0 days 00:03:12.471902,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.134940,NaN,NaN,20.0,NaN,NaN,LGBM,NaN,89.0,NaN,NaN,NaN,NaN,COMPLETE


In [24]:
study.trials_dataframe()['params_model'].value_counts()

params_model
RF      9
SVM     7
LGBM    5
KNN     3
XGB     3
GB      3
Name: count, dtype: int64

In [30]:
study.best_params

{'model': 'RF', 'n_estimators_rf': 134, 'max_depth_rf': 13}

In [33]:
best_xgb = RandomForestRegressor(n_estimators =  134, max_depth =  13)

model = TransformedTargetRegressor(regressor=best_xgb , transformer=pt)

model.fit(X_train_trans , y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",RandomForestR...stimators=134)
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",PowerTransformer()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,87
regressor_ regressor_: objectFitted regressor.,RandomForestRegressor,RandomForestR...stimators=134)
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,PowerTransformer,PowerTransformer()
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",134
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",13


In [34]:
y_pred = model.predict(X_test_trans)

r2_score(y_test , y_pred)

0.8281715609240685

In [35]:
study.trials_dataframe().head()

,number,value,datetime_start,datetime_complete,duration,params_c_linear,params_c_poly,params_c_rbf,params_degree_poly,params_gamma_rbf,params_kernel_svm,params_learning_rate_gb,params_learning_rate_lgbm,params_learning_rate_xgb,params_max_depth_gb,params_max_depth_lgbm,params_max_depth_rf,params_max_depth_xgb,params_model,params_n_estimators_gb,params_n_estimators_lgbm,params_n_estimators_rf,params_n_estimators_xgb,params_n_neighbors_knn,params_weights_knn,state
0,0,4.470294,2026-07-05 20:33:52.275037,2026-07-05 20:36:00.750280,0 days 00:02:08.475243,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KNN,NaN,NaN,NaN,NaN,12.0,distance,COMPLETE
1,1,3.047540,2026-07-05 20:33:52.275830,2026-07-05 20:35:08.088966,0 days 00:01:15.813136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.0,NaN,RF,NaN,NaN,134.0,NaN,NaN,NaN,COMPLETE
2,2,3.047806,2026-07-05 20:33:52.276544,2026-07-05 20:37:08.957801,0 days 00:03:16.681257,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.333788,NaN,NaN,11.0,NaN,NaN,LGBM,NaN,35.0,NaN,NaN,NaN,NaN,COMPLETE
3,3,4.567463,2026-07-05 20:33:52.276811,2026-07-05 20:44:22.579407,0 days 00:10:30.302596,5.874607,NaN,NaN,NaN,NaN,linear,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SVM,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,3.050770,2026-07-05 20:33:52.277376,2026-07-05 20:37:04.749278,0 days 00:03:12.471902,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.134940,NaN,NaN,20.0,NaN,NaN,LGBM,NaN,89.0,NaN,NaN,NaN,NaN,COMPLETE


In [36]:
plot_optimization_history(study)